# Automated Processed Refund Check

## Purpose

This notebook simulates an authorised accounting-system extract and checks whether refunds have been processed since cases entered the operational workbook.

Confirmed matches can be completed automatically, reducing unnecessary manual case reviews.

Cases with missing, partial, conflicting or ambiguous refund information remain available for an agent to investigate.

## Matching Approach

Refund transactions are matched to operational cases using both:

- Policy Number
- Client Number

Using both identifiers reduces the risk of matching a refund to the wrong policy when one client holds multiple policies.

A case can only be completed automatically when:

- the case is not already completed
- the policy number and client number match
- the refund was processed after the case snapshot date
- one refund transaction amount matches the outstanding amount
- the match is unique and unambiguous

Partial, missing, conflicting or multiple matches remain available for agent review.

The refund reason is retained for audit purposes but is not used as an automated matching rule.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

OPERATIONAL_DATA_DIR = PROJECT_ROOT / "data" / "operational"
ACCOUNTING_DATA_DIR = PROJECT_ROOT / "data" / "accounting"

SOURCE_CASES_PATH = OPERATIONAL_DATA_DIR / "source_cases.csv"
AGENT_UPDATES_PATH = OPERATIONAL_DATA_DIR / "agent_updates.csv"
ACCOUNTING_FEED_PATH = ACCOUNTING_DATA_DIR / "refund_transactions.csv"

ACCOUNTING_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
print(PROJECT_ROOT)
print(ACCOUNTING_DATA_DIR)

c:\Missed Refund Detection and Process Improvement
c:\Missed Refund Detection and Process Improvement\data\accounting


## Load Operational Data

The source case data and persistent agent updates are loaded separately.

This preserves the refresh-safe design established earlier in the project and allows the automated check to update operational records without changing the original source data.

In [4]:
source_cases = pd.read_csv(
    SOURCE_CASES_PATH,
    parse_dates=["Snapshot Date"]
)

agent_updates = pd.read_csv(
    AGENT_UPDATES_PATH,
    parse_dates=["Completion Date", "Last Updated Date"]
)

print("Source cases:", source_cases.shape)
print("Agent updates:", agent_updates.shape)

Source cases: (2871, 11)
Agent updates: (2871, 8)


In [5]:
print("Source case columns:")
print(source_cases.columns.tolist())

print("\nAgent update columns:")
print(agent_updates.columns.tolist())

Source case columns:
['Case ID', 'Policy Number', 'Client Number', 'Snapshot Date', 'Outstanding Amount', 'Cancellation Status', 'Cancelling Department', 'Cancelling Agent', 'Refund Type', 'Root Cause', 'Customer Contacted']

Agent update columns:
['Case ID', 'Agent Working', 'Agent Notes', 'Case Status', 'Final Outcome', 'Completed By', 'Completion Date', 'Last Updated Date']


In [6]:
print("Duplicate source Case IDs:", source_cases["Case ID"].duplicated().sum())
print("Duplicate update Case IDs:", agent_updates["Case ID"].duplicated().sum())

missing_update_records = (
    source_cases[["Case ID"]]
    .merge(
        agent_updates[["Case ID"]],
        on="Case ID",
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
)

print(
    "Source cases without an update record:",
    len(missing_update_records)
)

Duplicate source Case IDs: 0
Duplicate update Case IDs: 0
Source cases without an update record: 0


## Define the Stable Automation Test Population

The synthetic accounting scenarios must be generated from the same case population every time the notebook runs.

Cases completed manually are excluded because their operational outcome has already been confirmed.

Cases previously completed by the automated check remain in the test population. This keeps the generated accounting feed and audit results reproducible, while the later update controls prevent completed records from being changed again.

In [7]:
case_status_details = agent_updates[
    [
        "Case ID",
        "Case Status",
        "Completed By",
    ]
].copy()

automation_test_population = (
    source_cases
    .merge(
        case_status_details,
        on="Case ID",
        how="left",
        validate="one_to_one",
    )
    .loc[
        lambda df: ~(
            df["Case Status"].eq("Completed")
            & df["Completed By"].ne("Automated Check")
        )
    ]
    .copy()
)

eligible_cases = automation_test_population.copy()

print("Stable automation test population:", len(eligible_cases))

print("\nCurrent case status breakdown:")
print(agent_updates["Case Status"].value_counts(dropna=False))

print(
    "\nManually completed cases excluded:",
    (
        agent_updates["Case Status"].eq("Completed")
        & agent_updates["Completed By"].ne("Automated Check")
    ).sum()
)

print(
    "Previously automated cases retained:",
    agent_updates["Completed By"].eq("Automated Check").sum()
)

Stable automation test population: 2870

Current case status breakdown:
Case Status
Open         2370
Completed     501
Name: count, dtype: int64

Manually completed cases excluded: 1
Previously automated cases retained: 500


## Define Accounting Feed Structure

The simulated accounting extract uses one row per refund transaction.

Each transaction contains the identifiers and details that would be visible on the accounting screen:

- Refund Transaction ID
- Policy Number
- Client Number
- Refund Amount
- Refund Processed By
- Refund Processed Date
- Refund Reason

The synthetic feed will include a mixture of confirmed matches, partial matches, ambiguous matches and unrelated transactions so that the automated controls can be tested realistically.

In [8]:
ACCOUNTING_COLUMNS = [
    "Refund Transaction ID",
    "Policy Number",
    "Client Number",
    "Refund Amount",
    "Refund Processed By",
    "Refund Processed Date",
    "Refund Reason",
]

print(ACCOUNTING_COLUMNS)

['Refund Transaction ID', 'Policy Number', 'Client Number', 'Refund Amount', 'Refund Processed By', 'Refund Processed Date', 'Refund Reason']


## Set Reproducible Generation Rules

A fixed random seed ensures that the same synthetic accounting transactions are generated each time the notebook is run.

This is important for testing, debugging and portfolio reproducibility.

In [9]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Random seed:", RANDOM_SEED)

Random seed: 42


## Select Confirmed Refund Matches

A sample of eligible cases will be used to create valid accounting transactions.

These transactions will have:

- matching policy and client numbers
- a refund amount equal to the outstanding amount
- a processed date after the snapshot date

They represent cases that the automated process should safely complete.

In [10]:
confirmed_case_count = 500

confirmed_cases = (
    eligible_cases
    .sample(
        n=confirmed_case_count,
        random_state=RANDOM_SEED
    )
    .copy()
)

print("Confirmed-match cases selected:", len(confirmed_cases))
print("Duplicate selected Case IDs:", confirmed_cases["Case ID"].duplicated().sum())

Confirmed-match cases selected: 500
Duplicate selected Case IDs: 0


In [11]:
confirmed_transactions = confirmed_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].copy()

confirmed_transactions["Refund Transaction ID"] = [
    f"RT{i:06d}"
    for i in range(1, len(confirmed_transactions) + 1)
]

confirmed_transactions["Refund Amount"] = (
    confirmed_transactions["Outstanding Amount"]
)

confirmed_transactions["Refund Processed By"] = rng.choice(
    ["DH001", "DH002", "FIN001", "FIN002"],
    size=len(confirmed_transactions)
)

days_after_snapshot = rng.integers(
    low=1,
    high=31,
    size=len(confirmed_transactions)
)

confirmed_transactions["Refund Processed Date"] = (
    confirmed_transactions["Snapshot Date"]
    + pd.to_timedelta(days_after_snapshot, unit="D")
)

confirmed_transactions["Refund Reason"] = rng.choice(
    [
        "Missed cancellation refund",
        "Premium collected after cancellation",
        "Refund due following policy review",
        "Customer account correction",
    ],
    size=len(confirmed_transactions)
)

confirmed_transactions = confirmed_transactions[
    ACCOUNTING_COLUMNS
]

print("Confirmed transactions created:", len(confirmed_transactions))
print(confirmed_transactions.head())

Confirmed transactions created: 500
     Refund Transaction ID Policy Number Client Number  Refund Amount  \
444               RT000001     PET000445      CL000043          25.59   
2395              RT000002     PET002396      CL000156          47.49   
762               RT000003     PET000763      CL000053          16.41   
652               RT000004     PET000653      CL000124          97.63   
2675              RT000005     PET002676      CL000117          78.62   

     Refund Processed By Refund Processed Date  \
444                DH001            2025-04-04   
2395              FIN002            2026-04-20   
762               FIN001            2025-06-29   
652                DH002            2025-06-27   
2675               DH002            2026-06-09   

                           Refund Reason  
444          Customer account correction  
2395  Refund due following policy review  
762   Refund due following policy review  
652          Customer account correction  
2675     

In [12]:
confirmed_validation = confirmed_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].merge(
    confirmed_transactions,
    on=["Policy Number", "Client Number"],
    how="left",
    validate="one_to_one"
)

amount_matches = (
    confirmed_validation["Outstanding Amount"]
    .round(2)
    .eq(confirmed_validation["Refund Amount"].round(2))
)

processed_after_snapshot = (
    confirmed_validation["Refund Processed Date"]
    > confirmed_validation["Snapshot Date"]
)

print("Transactions with matching amounts:", amount_matches.sum())
print("Transactions processed after snapshot:", processed_after_snapshot.sum())
print("Missing transaction IDs:", confirmed_validation["Refund Transaction ID"].isna().sum())

Transactions with matching amounts: 500
Transactions processed after snapshot: 500
Missing transaction IDs: 0


## Prepare Exception-Test Cases

The remaining eligible cases will be used to create accounting transactions that should not be completed automatically.

Keeping these cases separate from the confirmed-match sample prevents one case from appearing in more than one test scenario.

In [13]:
remaining_cases = (
    eligible_cases
    .loc[
        ~eligible_cases["Case ID"].isin(
            confirmed_cases["Case ID"]
        )
    ]
    .copy()
)

print("Remaining eligible cases:", len(remaining_cases))
print(
    "Overlap with confirmed cases:",
    remaining_cases["Case ID"]
    .isin(confirmed_cases["Case ID"])
    .sum()
)

Remaining eligible cases: 2370
Overlap with confirmed cases: 0


## Create Amount-Mismatch Transactions

Some accounting transactions will match the correct policy and client but have a refund amount that does not equal the outstanding balance recorded in the operational case.

This does not mean that a partial refund was issued. Differences may exist because the accounting transaction and the snapshot balance represent different calculations, dates or account activity.

These cases must remain open for agent review because the automated process cannot safely confirm that the identified outstanding balance has been fully resolved.

In [14]:
amount_mismatch_case_count = 80

amount_mismatch_cases = (
    remaining_cases
    .sample(
        n=amount_mismatch_case_count,
        random_state=RANDOM_SEED + 1
    )
    .copy()
)

print("Amount-mismatch cases selected:", len(amount_mismatch_cases))
print(
    "Duplicate selected Case IDs:",
    amount_mismatch_cases["Case ID"].duplicated().sum()
)

Amount-mismatch cases selected: 80
Duplicate selected Case IDs: 0


In [15]:
amount_mismatch_transactions = amount_mismatch_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].copy()

amount_mismatch_transactions["Refund Transaction ID"] = [
    f"RT{i:06d}"
    for i in range(
        len(confirmed_transactions) + 1,
        len(confirmed_transactions) + len(amount_mismatch_transactions) + 1
    )
]

mismatch_multiplier = rng.choice(
    [0.75, 0.85, 1.10, 1.20],
    size=len(amount_mismatch_transactions)
)

amount_mismatch_transactions["Refund Amount"] = (
    amount_mismatch_transactions["Outstanding Amount"]
    * mismatch_multiplier
).round(2)

amount_mismatch_transactions["Refund Processed By"] = rng.choice(
    ["DH001", "DH002", "FIN001", "FIN002"],
    size=len(amount_mismatch_transactions)
)

days_after_snapshot = rng.integers(
    low=1,
    high=31,
    size=len(amount_mismatch_transactions)
)

amount_mismatch_transactions["Refund Processed Date"] = (
    amount_mismatch_transactions["Snapshot Date"]
    + pd.to_timedelta(days_after_snapshot, unit="D")
)

amount_mismatch_transactions["Refund Reason"] = rng.choice(
    [
        "Account balance correction",
        "Premium adjustment following review",
        "Refund processed after account recalculation",
        "Manual accounting correction",
    ],
    size=len(amount_mismatch_transactions)
)

amount_mismatch_transactions = amount_mismatch_transactions[
    ACCOUNTING_COLUMNS
]

print(
    "Amount-mismatch transactions created:",
    len(amount_mismatch_transactions)
)

print(amount_mismatch_transactions.head())

Amount-mismatch transactions created: 80
     Refund Transaction ID Policy Number Client Number  Refund Amount  \
2466              RT000501     PET002467      CL000038           2.57   
2110              RT000502     PET002111      CL000088         129.13   
2272              RT000503     PET002273      CL000040           8.26   
1791              RT000504     PET001792      CL000103          12.27   
288               RT000505     PET000289      CL000108          73.23   

     Refund Processed By Refund Processed Date  \
2466               DH002            2026-05-19   
2110               DH001            2026-03-19   
2272              FIN001            2026-03-06   
1791               DH002            2026-01-17   
288               FIN001            2025-03-16   

                                     Refund Reason  
2466           Premium adjustment following review  
2110  Refund processed after account recalculation  
2272           Premium adjustment following review  
1791   

In [16]:
amount_mismatch_validation = amount_mismatch_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].merge(
    amount_mismatch_transactions,
    on=["Policy Number", "Client Number"],
    how="left",
    validate="one_to_one"
)

amounts_different = (
    amount_mismatch_validation["Outstanding Amount"]
    .round(2)
    .ne(amount_mismatch_validation["Refund Amount"].round(2))
)

processed_after_snapshot = (
    amount_mismatch_validation["Refund Processed Date"]
    > amount_mismatch_validation["Snapshot Date"]
)

print("Transactions with different amounts:", amounts_different.sum())
print("Transactions processed after snapshot:", processed_after_snapshot.sum())
print(
    "Missing transaction IDs:",
    amount_mismatch_validation["Refund Transaction ID"].isna().sum()
)

Transactions with different amounts: 80
Transactions processed after snapshot: 80
Missing transaction IDs: 0


## Create Pre-Snapshot Refund Transactions

Some accounting transactions will match the policy number, client number and outstanding amount but will have been processed on or before the operational snapshot date.

These transactions must not be treated as evidence that the current missed-refund case has been resolved, because they may relate to earlier account activity.

In [17]:
remaining_after_amount_mismatch = (
    remaining_cases
    .loc[
        ~remaining_cases["Case ID"].isin(
            amount_mismatch_cases["Case ID"]
        )
    ]
    .copy()
)

pre_snapshot_case_count = 60

pre_snapshot_cases = (
    remaining_after_amount_mismatch
    .sample(
        n=pre_snapshot_case_count,
        random_state=RANDOM_SEED + 2
    )
    .copy()
)

print("Pre-snapshot cases selected:", len(pre_snapshot_cases))
print(
    "Overlap with amount-mismatch cases:",
    pre_snapshot_cases["Case ID"]
    .isin(amount_mismatch_cases["Case ID"])
    .sum()
)

Pre-snapshot cases selected: 60
Overlap with amount-mismatch cases: 0


In [18]:
pre_snapshot_transactions = pre_snapshot_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].copy()

start_id = (
    len(confirmed_transactions)
    + len(amount_mismatch_transactions)
    + 1
)

pre_snapshot_transactions["Refund Transaction ID"] = [
    f"RT{i:06d}"
    for i in range(
        start_id,
        start_id + len(pre_snapshot_transactions)
    )
]

pre_snapshot_transactions["Refund Amount"] = (
    pre_snapshot_transactions["Outstanding Amount"]
)

pre_snapshot_transactions["Refund Processed By"] = rng.choice(
    ["DH001", "DH002", "FIN001", "FIN002"],
    size=len(pre_snapshot_transactions)
)

days_before_snapshot = rng.integers(
    low=0,
    high=31,
    size=len(pre_snapshot_transactions)
)

pre_snapshot_transactions["Refund Processed Date"] = (
    pre_snapshot_transactions["Snapshot Date"]
    - pd.to_timedelta(days_before_snapshot, unit="D")
)

pre_snapshot_transactions["Refund Reason"] = rng.choice(
    [
        "Earlier policy refund",
        "Historic account correction",
        "Previous cancellation refund",
        "Prior premium adjustment",
    ],
    size=len(pre_snapshot_transactions)
)

pre_snapshot_transactions = pre_snapshot_transactions[
    ACCOUNTING_COLUMNS
]

print(
    "Pre-snapshot transactions created:",
    len(pre_snapshot_transactions)
)

print(pre_snapshot_transactions.head())

Pre-snapshot transactions created: 60
     Refund Transaction ID Policy Number Client Number  Refund Amount  \
1939              RT000581     PET001940      CL000073          96.08   
713               RT000582     PET000714      CL000086          96.59   
2260              RT000583     PET002261      CL000089          33.92   
48                RT000584     PET000049      CL000184           8.28   
2375              RT000585     PET002376      CL000002          10.99   

     Refund Processed By Refund Processed Date                Refund Reason  
1939               DH002            2026-01-20        Earlier policy refund  
713               FIN002            2025-05-23  Historic account correction  
2260               DH002            2026-02-23     Prior premium adjustment  
48                FIN002            2025-01-05        Earlier policy refund  
2375              FIN002            2026-03-12  Historic account correction  


In [19]:
pre_snapshot_validation = pre_snapshot_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].merge(
    pre_snapshot_transactions,
    on=["Policy Number", "Client Number"],
    how="left",
    validate="one_to_one"
)

amounts_match = (
    pre_snapshot_validation["Outstanding Amount"]
    .round(2)
    .eq(pre_snapshot_validation["Refund Amount"].round(2))
)

not_after_snapshot = (
    pre_snapshot_validation["Refund Processed Date"]
    <= pre_snapshot_validation["Snapshot Date"]
)

print("Transactions with matching amounts:", amounts_match.sum())
print("Transactions on or before snapshot:", not_after_snapshot.sum())
print(
    "Missing transaction IDs:",
    pre_snapshot_validation["Refund Transaction ID"].isna().sum()
)

Transactions with matching amounts: 60
Transactions on or before snapshot: 60
Missing transaction IDs: 0


## Create Ambiguous Multiple-Match Transactions

Some cases will have more than one accounting transaction matching the same policy number, client number and outstanding amount after the snapshot date.

These cases must not be completed automatically because the process cannot determine which transaction relates to the operational case with sufficient confidence.

They remain available for agent review as ambiguous matches.

In [20]:
remaining_after_pre_snapshot = (
    remaining_after_amount_mismatch
    .loc[
        ~remaining_after_amount_mismatch["Case ID"].isin(
            pre_snapshot_cases["Case ID"]
        )
    ]
    .copy()
)

ambiguous_case_count = 40

ambiguous_cases = (
    remaining_after_pre_snapshot
    .sample(
        n=ambiguous_case_count,
        random_state=RANDOM_SEED + 3
    )
    .copy()
)

print("Ambiguous cases selected:", len(ambiguous_cases))
print(
    "Overlap with pre-snapshot cases:",
    ambiguous_cases["Case ID"]
    .isin(pre_snapshot_cases["Case ID"])
    .sum()
)

Ambiguous cases selected: 40
Overlap with pre-snapshot cases: 0


In [21]:
ambiguous_base = ambiguous_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].copy()

ambiguous_transactions = pd.concat(
    [ambiguous_base, ambiguous_base],
    ignore_index=True
)

start_id = (
    len(confirmed_transactions)
    + len(amount_mismatch_transactions)
    + len(pre_snapshot_transactions)
    + 1
)

ambiguous_transactions["Refund Transaction ID"] = [
    f"RT{i:06d}"
    for i in range(
        start_id,
        start_id + len(ambiguous_transactions)
    )
]

ambiguous_transactions["Refund Amount"] = (
    ambiguous_transactions["Outstanding Amount"]
)

ambiguous_transactions["Refund Processed By"] = rng.choice(
    ["DH001", "DH002", "FIN001", "FIN002"],
    size=len(ambiguous_transactions)
)

days_after_snapshot = rng.integers(
    low=1,
    high=31,
    size=len(ambiguous_transactions)
)

ambiguous_transactions["Refund Processed Date"] = (
    ambiguous_transactions["Snapshot Date"]
    + pd.to_timedelta(days_after_snapshot, unit="D")
)

ambiguous_transactions["Refund Reason"] = rng.choice(
    [
        "Duplicate refund record for review",
        "Repeated accounting entry",
        "Potential duplicate transaction",
    ],
    size=len(ambiguous_transactions)
)

ambiguous_transactions = ambiguous_transactions[
    ACCOUNTING_COLUMNS
]

print("Ambiguous transactions created:", len(ambiguous_transactions))
print(
    "Transactions per selected policy/client:"
)
print(
    ambiguous_transactions
    .groupby(["Policy Number", "Client Number"])
    .size()
    .value_counts()
)

Ambiguous transactions created: 80
Transactions per selected policy/client:
2    40
Name: count, dtype: int64


In [22]:
ambiguous_validation = ambiguous_cases[
    [
        "Policy Number",
        "Client Number",
        "Outstanding Amount",
        "Snapshot Date",
    ]
].merge(
    ambiguous_transactions,
    on=["Policy Number", "Client Number"],
    how="left",
    validate="one_to_many"
)

ambiguous_validation["Amount Matches"] = (
    ambiguous_validation["Outstanding Amount"]
    .round(2)
    .eq(ambiguous_validation["Refund Amount"].round(2))
)

ambiguous_validation["Processed After Snapshot"] = (
    ambiguous_validation["Refund Processed Date"]
    > ambiguous_validation["Snapshot Date"]
)

match_counts = (
    ambiguous_validation
    .groupby(["Policy Number", "Client Number"])
    ["Refund Transaction ID"]
    .count()
)

print(
    "Transactions with matching amounts:",
    ambiguous_validation["Amount Matches"].sum()
)

print(
    "Transactions processed after snapshot:",
    ambiguous_validation["Processed After Snapshot"].sum()
)

print(
    "Cases with exactly two matching transactions:",
    match_counts.eq(2).sum()
)

Transactions with matching amounts: 80
Transactions processed after snapshot: 80
Cases with exactly two matching transactions: 40


## Create Unrelated Accounting Transactions

The accounting extract may contain refund transactions that do not relate to any case currently awaiting review.

These records help test that the automated process does not create false matches simply because unrelated refund activity exists in the same extract.

In [23]:
unrelated_transaction_count = 100

unrelated_transactions = pd.DataFrame(
    {
        "Policy Number": [
            f"OTHER{i:06d}"
            for i in range(1, unrelated_transaction_count + 1)
        ],
        "Client Number": [
            f"EXT{i:06d}"
            for i in range(1, unrelated_transaction_count + 1)
        ],
        "Refund Amount": rng.uniform(
            0.50,
            300.00,
            size=unrelated_transaction_count
        ).round(2),
        "Refund Processed By": rng.choice(
            ["DH001", "DH002", "FIN001", "FIN002"],
            size=unrelated_transaction_count
        ),
        "Refund Processed Date": pd.to_datetime("2025-01-01")
        + pd.to_timedelta(
            rng.integers(
                low=0,
                high=550,
                size=unrelated_transaction_count
            ),
            unit="D"
        ),
        "Refund Reason": rng.choice(
            [
                "Unrelated policy refund",
                "Account correction",
                "Premium adjustment",
                "Historic cancellation refund",
            ],
            size=unrelated_transaction_count
        ),
    }
)

start_id = (
    len(confirmed_transactions)
    + len(amount_mismatch_transactions)
    + len(pre_snapshot_transactions)
    + len(ambiguous_transactions)
    + 1
)

unrelated_transactions["Refund Transaction ID"] = [
    f"RT{i:06d}"
    for i in range(
        start_id,
        start_id + len(unrelated_transactions)
    )
]

unrelated_transactions = unrelated_transactions[
    ACCOUNTING_COLUMNS
]

print("Unrelated transactions created:", len(unrelated_transactions))

matching_policy_numbers = unrelated_transactions[
    "Policy Number"
].isin(source_cases["Policy Number"]).sum()

matching_client_numbers = unrelated_transactions[
    "Client Number"
].isin(source_cases["Client Number"]).sum()

print("Policy numbers found in source cases:", matching_policy_numbers)
print("Client numbers found in source cases:", matching_client_numbers)

Unrelated transactions created: 100
Policy numbers found in source cases: 0
Client numbers found in source cases: 0


## Combine the Accounting Feed

All transaction scenarios are combined into one accounting extract.

The completed feed contains:

- confirmed matches
- amount mismatches
- pre-snapshot transactions
- ambiguous multiple matches
- unrelated accounting activity

Combining them allows the matching process to be tested against realistic valid and invalid transaction patterns.

In [24]:
refund_transactions = pd.concat(
    [
        confirmed_transactions,
        amount_mismatch_transactions,
        pre_snapshot_transactions,
        ambiguous_transactions,
        unrelated_transactions,
    ],
    ignore_index=True
)

refund_transactions = (
    refund_transactions
    .sort_values("Refund Transaction ID")
    .reset_index(drop=True)
)

print("Total accounting transactions:", len(refund_transactions))
print(
    "Duplicate transaction IDs:",
    refund_transactions["Refund Transaction ID"].duplicated().sum()
)
print("\nAccounting feed columns:")
print(refund_transactions.columns.tolist())

Total accounting transactions: 820
Duplicate transaction IDs: 0

Accounting feed columns:
['Refund Transaction ID', 'Policy Number', 'Client Number', 'Refund Amount', 'Refund Processed By', 'Refund Processed Date', 'Refund Reason']


## Save the Accounting Extract

The simulated accounting transactions are saved as a separate CSV file.

Keeping the accounting feed separate from the operational case data reflects the real process, where refund information comes from a different authorised system.

In [25]:
refund_transactions.to_csv(
    ACCOUNTING_FEED_PATH,
    index=False,
    date_format="%Y-%m-%d"
)

print("Accounting feed saved to:")
print(ACCOUNTING_FEED_PATH)
print("Rows saved:", len(refund_transactions))

Accounting feed saved to:
c:\Missed Refund Detection and Process Improvement\data\accounting\refund_transactions.csv
Rows saved: 820


## Reload and Validate the Accounting Extract

The saved accounting extract is reloaded from CSV before matching begins.

This confirms that the automation works from the persisted input file rather than relying only on data held temporarily in the notebook.

In [26]:
accounting_feed = pd.read_csv(
    ACCOUNTING_FEED_PATH,
    parse_dates=["Refund Processed Date"]
)

print("Reloaded accounting rows:", len(accounting_feed))
print("Reloaded accounting columns:", len(accounting_feed.columns))
print(
    "Duplicate transaction IDs:",
    accounting_feed["Refund Transaction ID"].duplicated().sum()
)
print(
    "Missing required values:",
    accounting_feed[ACCOUNTING_COLUMNS].isna().sum().sum()
)
print(
    "Refund processed date type:",
    accounting_feed["Refund Processed Date"].dtype
)

Reloaded accounting rows: 820
Reloaded accounting columns: 7
Duplicate transaction IDs: 0
Missing required values: 0
Refund processed date type: datetime64[us]


## Match Eligible Cases to Accounting Transactions

Eligible operational cases are joined to the accounting feed using both the policy number and client number.

The join initially retains all possible accounting matches. Each possible match is then tested against the amount and processed-date rules before the automation decides whether the case can be completed safely.

In [27]:
candidate_matches = eligible_cases.merge(
    accounting_feed,
    on=["Policy Number", "Client Number"],
    how="left",
    validate="one_to_many",
    indicator=True,
)

print("Eligible cases:", len(eligible_cases))
print("Candidate-match rows:", len(candidate_matches))

print("\nJoin result:")
print(candidate_matches["_merge"].value_counts())

Eligible cases: 2870
Candidate-match rows: 2910

Join result:
_merge
left_only     2190
both           720
right_only       0
Name: count, dtype: int64


## Evaluate Transaction Matching Rules

Each candidate accounting transaction is tested against the automated matching rules.

A transaction is considered a valid match only when:

- an accounting transaction exists
- the refund amount equals the outstanding amount
- the refund processed date is after the snapshot date

These checks are applied at transaction level before determining whether each case has one unique valid match.

In [28]:
candidate_matches["Transaction Found"] = (
    candidate_matches["_merge"] == "both"
)

candidate_matches["Amount Matches"] = (
    candidate_matches["Transaction Found"]
    & candidate_matches["Outstanding Amount"]
        .round(2)
        .eq(candidate_matches["Refund Amount"].round(2))
)

candidate_matches["Processed After Snapshot"] = (
    candidate_matches["Transaction Found"]
    & (
        candidate_matches["Refund Processed Date"]
        > candidate_matches["Snapshot Date"]
    )
)

candidate_matches["Valid Transaction Match"] = (
    candidate_matches["Transaction Found"]
    & candidate_matches["Amount Matches"]
    & candidate_matches["Processed After Snapshot"]
)

print(
    "Transactions found:",
    candidate_matches["Transaction Found"].sum()
)

print(
    "Transactions with matching amounts:",
    candidate_matches["Amount Matches"].sum()
)

print(
    "Transactions processed after snapshot:",
    candidate_matches["Processed After Snapshot"].sum()
)

print(
    "Valid transaction matches:",
    candidate_matches["Valid Transaction Match"].sum()
)

Transactions found: 720
Transactions with matching amounts: 640
Transactions processed after snapshot: 660
Valid transaction matches: 580


## Summarise Matches at Case Level

Transaction-level results are grouped by Case ID so the automation can determine how many valid refund matches each operational case has.

A case can only be completed automatically when it has exactly one valid transaction match.

Cases with no valid match or more than one valid match remain open for agent review.

In [29]:
case_match_summary = (
    candidate_matches
    .groupby("Case ID", as_index=False)
    .agg(
        Transactions_Found=("Transaction Found", "sum"),
        Valid_Match_Count=("Valid Transaction Match", "sum"),
    )
)

case_match_summary["Automation Decision"] = np.select(
    [
        case_match_summary["Valid_Match_Count"].eq(1),
        case_match_summary["Valid_Match_Count"].gt(1),
    ],
    [
        "Auto Complete",
        "Ambiguous Match",
    ],
    default="Agent Review",
)

print("Case-level rows:", len(case_match_summary))

print("\nAutomation decisions:")
print(case_match_summary["Automation Decision"].value_counts())

Case-level rows: 2870

Automation decisions:
Automation Decision
Agent Review       2330
Auto Complete       500
Ambiguous Match      40
Name: count, dtype: int64


## Assign Detailed Review Reasons

Cases that cannot be completed automatically are assigned a clear reason.

This supports operational reporting and helps agents understand why a case still requires investigation.

Possible results include:

- no accounting transaction found
- refund amount does not match the outstanding balance
- refund was processed on or before the snapshot date
- multiple valid refund transactions were found
- one unique valid match was confirmed

In [30]:
case_rule_summary = (
    candidate_matches
    .groupby("Case ID", as_index=False)
    .agg(
        Transactions_Found=("Transaction Found", "sum"),
        Amount_Match_Count=("Amount Matches", "sum"),
        Post_Snapshot_Count=("Processed After Snapshot", "sum"),
        Valid_Match_Count=("Valid Transaction Match", "sum"),
    )
)

case_rule_summary["Review Result"] = np.select(
    [
        case_rule_summary["Valid_Match_Count"].eq(1),
        case_rule_summary["Valid_Match_Count"].gt(1),
        case_rule_summary["Transactions_Found"].eq(0),
        case_rule_summary["Amount_Match_Count"].eq(0),
        case_rule_summary["Post_Snapshot_Count"].eq(0),
    ],
    [
        "Confirmed Unique Match",
        "Multiple Valid Matches",
        "No Accounting Transaction",
        "Amount Mismatch",
        "Refund Not After Snapshot",
    ],
    default="Agent Review Required",
)

print("Case-level rows:", len(case_rule_summary))

print("\nReview result breakdown:")
print(case_rule_summary["Review Result"].value_counts())

Case-level rows: 2870

Review result breakdown:
Review Result
No Accounting Transaction    2190
Confirmed Unique Match        500
Amount Mismatch                80
Refund Not After Snapshot      60
Multiple Valid Matches         40
Name: count, dtype: int64


## Extract Confirmed Transactions

Only cases with exactly one valid transaction match are selected for automated completion.

The matched transaction details are retained so the automated update has a clear audit trail showing:

- which refund transaction was used
- who processed the refund
- when it was processed
- the refund reason recorded in the accounting extract

In [31]:
confirmed_match_ids = case_rule_summary.loc[
    case_rule_summary["Review Result"].eq("Confirmed Unique Match"),
    "Case ID",
]

confirmed_match_details = (
    candidate_matches
    .loc[
        candidate_matches["Case ID"].isin(confirmed_match_ids)
        & candidate_matches["Valid Transaction Match"]
    ]
    [
        [
            "Case ID",
            "Refund Transaction ID",
            "Refund Amount",
            "Refund Processed By",
            "Refund Processed Date",
            "Refund Reason",
        ]
    ]
    .copy()
)

print("Confirmed match detail rows:", len(confirmed_match_details))
print(
    "Duplicate confirmed Case IDs:",
    confirmed_match_details["Case ID"].duplicated().sum()
)
print(
    "Duplicate confirmed transaction IDs:",
    confirmed_match_details["Refund Transaction ID"].duplicated().sum()
)

Confirmed match detail rows: 500
Duplicate confirmed Case IDs: 0
Duplicate confirmed transaction IDs: 0


## Create the Automated Check Results Table

A case-level results table records the outcome of the automated accounting check for every eligible case.

This provides an audit trail separate from the persistent agent updates and supports later reporting on:

- automated completions
- unmatched cases
- amount mismatches
- pre-snapshot refunds
- ambiguous matches

In [32]:
automated_check_results = (
    eligible_cases[
        [
            "Case ID",
            "Policy Number",
            "Client Number",
            "Snapshot Date",
            "Outstanding Amount",
        ]
    ]
    .merge(
        case_rule_summary,
        on="Case ID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        confirmed_match_details,
        on="Case ID",
        how="left",
        validate="one_to_one"
    )
)

automated_check_results["Automated Completion"] = (
    automated_check_results["Review Result"]
    .eq("Confirmed Unique Match")
)

print("Automated check result rows:", len(automated_check_results))
print(
    "Duplicate Case IDs:",
    automated_check_results["Case ID"].duplicated().sum()
)
print(
    "Automated completions:",
    automated_check_results["Automated Completion"].sum()
)

print("\nReview result breakdown:")
print(
    automated_check_results["Review Result"]
    .value_counts()
)

Automated check result rows: 2870
Duplicate Case IDs: 0
Automated completions: 500

Review result breakdown:
Review Result
No Accounting Transaction    2190
Confirmed Unique Match        500
Amount Mismatch                80
Refund Not After Snapshot      60
Multiple Valid Matches         40
Name: count, dtype: int64


## Validate the Automated Check Results

Before any operational records are updated, the results table is checked to confirm that:

- every eligible case appears once
- all automated completions have a supporting transaction
- cases not completed automatically do not contain a confirmed transaction ID
- the decision totals reconcile to the eligible-case population

In [33]:
auto_complete_mask = automated_check_results[
    "Automated Completion"
]

non_auto_complete_mask = ~auto_complete_mask

print(
    "Automated completions missing transaction IDs:",
    automated_check_results.loc[
        auto_complete_mask,
        "Refund Transaction ID"
    ].isna().sum()
)

print(
    "Non-automated cases with transaction IDs:",
    automated_check_results.loc[
        non_auto_complete_mask,
        "Refund Transaction ID"
    ].notna().sum()
)

print(
    "Decision total:",
    automated_check_results["Review Result"]
    .value_counts()
    .sum()
)

print(
    "Eligible-case total:",
    len(eligible_cases)
)

Automated completions missing transaction IDs: 0
Non-automated cases with transaction IDs: 0
Decision total: 2870
Eligible-case total: 2870


## Save the Automated Check Audit Results

The case-level automated check results are saved before any persistent operational records are changed.

This creates a clear audit trail showing how every eligible case was assessed and which accounting transaction supported each automated completion.

In [34]:
AUTOMATED_RESULTS_PATH = (
    ACCOUNTING_DATA_DIR
    / "automated_refund_check_results.csv"
)

automated_check_results.to_csv(
    AUTOMATED_RESULTS_PATH,
    index=False,
    date_format="%Y-%m-%d",
)

print("Automated check results saved to:")
print(AUTOMATED_RESULTS_PATH)
print("Rows saved:", len(automated_check_results))

Automated check results saved to:
c:\Missed Refund Detection and Process Improvement\data\accounting\automated_refund_check_results.csv
Rows saved: 2870


## Prepare Persistent Agent Updates

A copy of the persistent agent update table is prepared before applying automated completions.

Only cases that:

- have one confirmed unique accounting match
- are not already completed

may be changed.

Existing agent assignments and notes are preserved. The separate automated-check results file retains the detailed accounting audit trail.

In [35]:
updated_agent_updates = agent_updates.copy()

auto_complete_case_ids = set(
    automated_check_results.loc[
        automated_check_results["Automated Completion"],
        "Case ID",
    ]
)

auto_update_mask = (
    updated_agent_updates["Case ID"].isin(auto_complete_case_ids)
    & updated_agent_updates["Case Status"].ne("Completed")
)

print("Cases approved for automated completion:", len(auto_complete_case_ids))
print("Persistent records eligible to update:", auto_update_mask.sum())

print(
    "Approved cases already completed:",
    (
        updated_agent_updates["Case ID"].isin(auto_complete_case_ids)
        & updated_agent_updates["Case Status"].eq("Completed")
    ).sum()
)

Cases approved for automated completion: 500
Persistent records eligible to update: 0
Approved cases already completed: 500


In [36]:
existing_auto_case_ids = set(
    agent_updates.loc[
        agent_updates["Completed By"].eq("Automated Check"),
        "Case ID",
    ]
)

if existing_auto_case_ids:
    assert existing_auto_case_ids == auto_complete_case_ids, (
        "Safety stop: generated automated cases do not match "
        "the cases already completed by automation."
    )

    assert auto_update_mask.sum() == 0, (
        "Safety stop: existing automated cases were unexpectedly "
        "identified for another update."
    )

print("Automation rerun safety check passed.")

Automation rerun safety check passed.


## Prepare Automated Completion Details

The confirmed accounting matches are converted into a Case ID lookup before updating the persistent operational table.

For automated completions:

- Case Status becomes Completed
- Final Outcome becomes Refund Already Processed
- Completed By records Automated Check rather than attributing the review to an agent
- Completion Date uses the date the refund was processed
- Last Updated Date records the date the automation updated the operational record

Agent assignments and notes are not overwritten.

In [37]:
auto_completion_lookup = (
    automated_check_results
    .loc[
        automated_check_results["Automated Completion"],
        [
            "Case ID",
            "Refund Transaction ID",
            "Refund Processed Date",
        ],
    ]
    .set_index("Case ID")
)

automation_run_date = pd.Timestamp.today().normalize()

print("Completion lookup rows:", len(auto_completion_lookup))
print(
    "Duplicate lookup Case IDs:",
    auto_completion_lookup.index.duplicated().sum()
)
print(
    "Missing processed dates:",
    auto_completion_lookup["Refund Processed Date"].isna().sum()
)
print("Automation run date:", automation_run_date.date())

Completion lookup rows: 500
Duplicate lookup Case IDs: 0
Missing processed dates: 0
Automation run date: 2026-07-23


In [38]:
updated_agent_updates.loc[
    auto_update_mask,
    "Case Status"
] = "Completed"

updated_agent_updates.loc[
    auto_update_mask,
    "Final Outcome"
] = "Refund Already Processed"

updated_agent_updates.loc[
    auto_update_mask,
    "Completed By"
] = "Automated Check"

updated_agent_updates.loc[
    auto_update_mask,
    "Completion Date"
] = (
    updated_agent_updates.loc[
        auto_update_mask,
        "Case ID"
    ]
    .map(
        auto_completion_lookup[
            "Refund Processed Date"
        ]
    )
)

updated_agent_updates.loc[
    auto_update_mask,
    "Last Updated Date"
] = automation_run_date

print(
    "Completed cases after automated update:",
    updated_agent_updates["Case Status"]
    .eq("Completed")
    .sum()
)

print(
    "Cases completed by automated check:",
    updated_agent_updates["Completed By"]
    .eq("Automated Check")
    .sum()
)

print(
    "Automated completions missing completion dates:",
    updated_agent_updates.loc[
        updated_agent_updates["Completed By"]
        .eq("Automated Check"),
        "Completion Date"
    ]
    .isna()
    .sum()
)

Completed cases after automated update: 501
Cases completed by automated check: 500
Automated completions missing completion dates: 0


## Validate Persistent Updates

The updated persistent table is compared with the original version before saving.

This confirms that:

- exactly 500 new cases were completed automatically
- the existing completed case remains unchanged
- agent assignments and notes were preserved
- every automated completion has the correct outcome, completion source and dates

In [39]:
comparison = agent_updates.merge(
    updated_agent_updates,
    on="Case ID",
    how="inner",
    suffixes=("_Before", "_After"),
    validate="one_to_one",
)

agent_assignments_changed = (
    comparison["Agent Working_Before"].fillna("")
    != comparison["Agent Working_After"].fillna("")
).sum()

agent_notes_changed = (
    comparison["Agent Notes_Before"].fillna("")
    != comparison["Agent Notes_After"].fillna("")
).sum()

newly_completed = (
    comparison["Case Status_Before"].ne("Completed")
    & comparison["Case Status_After"].eq("Completed")
).sum()

existing_completed_changed = (
    comparison["Case Status_Before"].eq("Completed")
    & (
        comparison["Final Outcome_Before"].fillna("")
        != comparison["Final Outcome_After"].fillna("")
    )
).sum()

automated_records = updated_agent_updates[
    updated_agent_updates["Completed By"].eq("Automated Check")
]

print("Newly completed cases:", newly_completed)
print("Agent assignments changed:", agent_assignments_changed)
print("Agent notes changed:", agent_notes_changed)
print("Existing completed outcomes changed:", existing_completed_changed)

print(
    "Automated records with incorrect outcome:",
    automated_records["Final Outcome"]
    .ne("Refund Already Processed")
    .sum()
)

print(
    "Automated records missing last-updated dates:",
    automated_records["Last Updated Date"]
    .isna()
    .sum()
)

Newly completed cases: 0
Agent assignments changed: 0
Agent notes changed: 0
Existing completed outcomes changed: 0
Automated records with incorrect outcome: 0
Automated records missing last-updated dates: 0


## Save Updated Persistent Agent Records

The validated persistent update table is saved only after all safeguards have passed.

This updates the operational backend with the confirmed automated completions while preserving agent-entered information and the existing refresh-safe structure.

In [40]:
updated_agent_updates.to_csv(
    AGENT_UPDATES_PATH,
    index=False,
    date_format="%Y-%m-%d",
)

print("Updated agent records saved to:")
print(AGENT_UPDATES_PATH)
print("Rows saved:", len(updated_agent_updates))

Updated agent records saved to:
c:\Missed Refund Detection and Process Improvement\data\operational\agent_updates.csv
Rows saved: 2871


## Reload and Validate Saved Agent Updates

The updated persistent agent table is reloaded from CSV after saving.

This confirms that the automated completions were written correctly to disk and that the saved file remains structurally valid.

In [42]:
saved_agent_updates = pd.read_csv(
    AGENT_UPDATES_PATH,
    parse_dates=["Completion Date", "Last Updated Date"],
)

print("Reloaded rows:", len(saved_agent_updates))
print(
    "Duplicate Case IDs:",
    saved_agent_updates["Case ID"].duplicated().sum()
)

print(
    "Completed cases:",
    saved_agent_updates["Case Status"]
    .eq("Completed")
    .sum()
)

print(
    "Cases completed by automated check:",
    saved_agent_updates["Completed By"]
    .eq("Automated Check")
    .sum()
)

print(
    "Automated completions missing completion dates:",
    saved_agent_updates.loc[
        saved_agent_updates["Completed By"].eq("Automated Check"),
        "Completion Date",
    ]
    .isna()
    .sum()
)

print(
    "Automated completions missing last-updated dates:",
    saved_agent_updates.loc[
        saved_agent_updates["Completed By"].eq("Automated Check"),
        "Last Updated Date",
    ]
    .isna()
    .sum()
)

Reloaded rows: 2871
Duplicate Case IDs: 0
Completed cases: 501
Cases completed by automated check: 500
Automated completions missing completion dates: 0
Automated completions missing last-updated dates: 0


## Final Validation Summary

The automated processed-refund check completed successfully.

Results:

- 2,870 cases were included in the stable automation test population
- 820 accounting transactions were generated
- 500 cases had one confirmed unique refund match
- 80 cases had an amount mismatch
- 60 cases had a refund processed on or before the snapshot date
- 40 cases had multiple valid matches
- 2,190 cases had no matching accounting transaction
- 500 cases were completed automatically
- existing agent assignments and notes were preserved
- previously completed cases were protected
- rerunning the notebook created no duplicate updates

The workflow is therefore reproducible, auditable and safe to rerun.